# 04 — Model Optimisation & Benchmarking

**Pipeline:**
1. Load existing ONNX models
2. Graph optimisation (ONNX Runtime)
3. FP16 conversion
4. Optional INT8 dynamic quantisation
5. Accuracy sanity-check after quantisation
6. Latency + throughput benchmark (CPU)
7. Memory & size comparison table


In [ ]:
import time, os, gc
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
import matplotlib.pyplot as plt
import pandas as pd

ROOT       = Path("..")
MODELS_DIR = ROOT / "models"
OPT_DIR    = MODELS_DIR / "optimised"
OPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model paths ───────────────────────────────────────────────
MODELS = {
    "static_medium":   MODELS_DIR / "arabic_sign_model_medium.onnx",
    "dynamic_large":   MODELS_DIR / "arabic_sign_video_model_large.onnx",
    "dynamic_small":   MODELS_DIR / "arabic_sign_video_model_small.onnx",
}

for name, path in MODELS.items():
    exists = path.exists()
    size   = f"{path.stat().st_size / 1e6:.1f} MB" if exists else "NOT FOUND"
    print(f"  {name:<20} {size}")

In [ ]:
# ── Session factory ───────────────────────────────────────────
def make_session(model_path: str, opt_level=ort.GraphOptimizationLevel.ORT_ENABLE_ALL):
    opts = ort.SessionOptions()
    opts.graph_optimization_level = opt_level
    opts.inter_op_num_threads = 4
    opts.intra_op_num_threads = 4
    return ort.InferenceSession(model_path, sess_options=opts,
                                 providers=["CPUExecutionProvider"])

In [ ]:
# ── Graph Optimisation (ORT built-in) ─────────────────────────
# ORT's ORT_ENABLE_ALL applies constant folding, operator fusion,
# layout rewriting, and memory planning — no file needed.

for name, path in MODELS.items():
    if not path.exists():
        print(f"  [SKIP] {name}")
        continue
    sess = make_session(str(path))
    print(f"  {name}: session loaded with ORT_ENABLE_ALL ✓")

In [ ]:
# ── FP16 Conversion ───────────────────────────────────────────
from onnxmltools.utils import load_model, save_model
from onnxmltools.utils.float16_converter import convert_float_to_float16

fp16_paths = {}

for name, path in MODELS.items():
    if not path.exists():
        continue
    try:
        model_fp32 = load_model(str(path))
        model_fp16 = convert_float_to_float16(model_fp32, keep_io_types=True)
        out_path   = OPT_DIR / path.name.replace(".onnx", "_fp16.onnx")
        save_model(model_fp16, str(out_path))
        fp16_paths[name] = out_path
        orig_mb  = path.stat().st_size / 1e6
        fp16_mb  = out_path.stat().st_size / 1e6
        print(f"  {name}: {orig_mb:.1f} MB → {fp16_mb:.1f} MB (FP16, {100*(1-fp16_mb/orig_mb):.0f}% smaller)")
    except ImportError:
        print("  onnxmltools not installed — pip install onnxmltools")
        break
    except Exception as e:
        print(f"  FP16 conversion failed for {name}: {e}")

In [ ]:
# ── INT8 Dynamic Quantisation (optional) ─────────────────────
int8_paths = {}

for name, path in MODELS.items():
    if not path.exists():
        continue
    out_path = OPT_DIR / path.name.replace(".onnx", "_int8.onnx")
    try:
        quantize_dynamic(
            model_input   = str(path),
            model_output  = str(out_path),
            weight_type   = QuantType.QInt8,
            optimize_model= True,
        )
        int8_paths[name] = out_path
        orig_mb = path.stat().st_size / 1e6
        int8_mb = out_path.stat().st_size / 1e6
        print(f"  {name}: {orig_mb:.1f} MB → {int8_mb:.1f} MB (INT8, {100*(1-int8_mb/orig_mb):.0f}% smaller)")
    except Exception as e:
        print(f"  INT8 failed for {name}: {e}")

In [ ]:
# ── Benchmark helper ──────────────────────────────────────────
def benchmark_session(sess: ort.InferenceSession,
                      dummy_input: dict,
                      n_warmup: int = 20,
                      n_runs:   int = 200) -> dict:
    # Warmup
    for _ in range(n_warmup):
        sess.run(None, dummy_input)

    # Timed runs
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        sess.run(None, dummy_input)
        times.append((time.perf_counter() - t0) * 1000)  # ms

    times = np.array(times)
    return {
        "mean_ms":   float(times.mean()),
        "p50_ms":    float(np.percentile(times, 50)),
        "p95_ms":    float(np.percentile(times, 95)),
        "p99_ms":    float(np.percentile(times, 99)),
        "fps":       1000.0 / times.mean(),
    }

In [ ]:
# ── Run benchmarks ────────────────────────────────────────────
STATIC_DUMMY  = {"landmarks": np.zeros((1, 126), dtype=np.float32)}
DYNAMIC_DUMMY = {"sequence":  np.zeros((1, 30, 126), dtype=np.float32)}

results = []

all_paths = {**{f"{k}_fp32": v for k, v in MODELS.items()},
             **{f"{k}_fp16": v for k, v in fp16_paths.items()},
             **{f"{k}_int8": v for k, v in int8_paths.items()}}

for label, path in all_paths.items():
    if not path.exists():
        continue
    try:
        sess   = make_session(str(path))
        inp    = sess.get_inputs()[0]
        dummy  = STATIC_DUMMY if "static" in label else DYNAMIC_DUMMY
        # Adapt dummy key to model's actual input name
        dummy  = {inp.name: list(dummy.values())[0]}
        stats  = benchmark_session(sess, dummy)
        size_mb = path.stat().st_size / 1e6
        results.append({"model": label, "size_MB": round(size_mb, 2), **{k: round(v, 2) for k, v in stats.items()}})
        print(f"  {label:<35} mean {stats['mean_ms']:.1f} ms | {stats['fps']:.0f} FPS")
    except Exception as e:
        print(f"  {label}: benchmark error — {e}")

In [ ]:
# ── Results table ─────────────────────────────────────────────
if results:
    df = pd.DataFrame(results).set_index("model")
    display(df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    df["mean_ms"].plot(kind="bar", ax=axes[0], color="steelblue", title="Mean Latency (ms)")
    df["fps"].plot(     kind="bar", ax=axes[1], color="darkorange", title="Throughput (FPS)")
    for ax in axes:
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.savefig(str(ROOT / "data" / "processed" / "benchmark_results.png"), dpi=120)
    plt.show()
else:
    print("No benchmark results — ensure models are present in models/ directory")

In [ ]:
# ── Accuracy sanity-check post-quantisation ───────────────────
# Compare FP32 vs INT8 logit outputs on random inputs

def logit_cosine_similarity(path_fp32, path_int8, dummy_input, n=50):
    if not (Path(path_fp32).exists() and Path(path_int8).exists()):
        return None
    s32  = make_session(path_fp32)
    s8   = make_session(path_int8)
    inp  = s32.get_inputs()[0]
    sims = []
    for _ in range(n):
        x   = {inp.name: list(dummy_input.values())[0] + np.random.randn(*list(dummy_input.values())[0].shape).astype(np.float32) * 0.1}
        o32 = s32.run(None, x)[0][0]
        o8  = s8.run(None,  x)[0][0]
        cos = np.dot(o32, o8) / (np.linalg.norm(o32) * np.linalg.norm(o8) + 1e-9)
        sims.append(cos)
    return float(np.mean(sims))

for name in MODELS:
    fp32_p = MODELS[name]
    int8_p = int8_paths.get(name)
    if fp32_p.exists() and int8_p and int8_p.exists():
        dummy = STATIC_DUMMY if "static" in name else DYNAMIC_DUMMY
        sim   = logit_cosine_similarity(str(fp32_p), str(int8_p), dummy)
        print(f"  {name}: FP32 vs INT8 logit cosine similarity = {sim:.4f}")

## Summary

| Format | Typical size reduction | Speed-up |
|---|---|---|
| FP32 (baseline) | — | 1× |
| FP16 | ~50% | 1.5–2× |
| INT8 dynamic | ~75% | 2–4× |

> For deployment, copy the best `.onnx` file to `models/` and update `backend/config.py`.
